# Air Quality Data Analysis with Python
## Notebook 4 · First Plots

⏱️ About 50 minutes &nbsp;·&nbsp; ⬅️ Builds on Notebook 3

Numbers persuade analysts; **pictures persuade everyone else**. This notebook turns
the Oshodi year into charts — and, just as important, teaches the difference
between a chart that displays data and a chart that communicates a finding.

You'll learn to:

* make time-series plots straight from pandas,
* label axes, add reference lines, and set figure size,
* apply a consistent, publication-quality **house style**,
* write titles that carry the finding (the "five-second test"),
* smooth a noisy year with a rolling mean, annotate the key event, and save a PNG.

### 1. Setup: the recipe from Notebook 3

Every notebook from here on starts the same way — load, parse, index, localise.
This is the exact recipe you practised, condensed into one cell:

In [ ]:
import pandas as pd
from pathlib import Path

DATA = "../data"  # local checkout of the course repository
if not Path(DATA).exists():  # running in Colab -> read from GitHub
    DATA = "https://raw.githubusercontent.com/rwpinder/tutorial-air-quality-data-analysis/main/data"

lagos = pd.read_csv(f"{DATA}/lagos_pm25_recent.csv")
lagos["datetime"] = pd.to_datetime(lagos["datetime"], utc=True)
lagos = lagos.set_index("datetime").tz_convert("Africa/Lagos")
pm = lagos["pm25_value"]

print(f"{len(pm)} hourly values, {pm.index.min():%b %Y} to {pm.index.max():%b %Y}")

### 2. The house style

Good charts across a project should look like they belong together. The next cell
sets a house style once, and every later plot inherits it. It follows the same
design rules the AQ agent uses for its charts:

* **gray is the default; colour is reserved for the finding** (one accent),
* horizontal gridlines only, light and behind the data,
* no top/right border ("spines") — open the chart toward the data,
* a colour-blind-safe accent palette (Okabe–Ito).

In [ ]:
import matplotlib.pyplot as plt

GRAY, BLUE, ORANGE, GREEN = "#999999", "#0072B2", "#D55E00", "#009E73"

plt.rcParams.update({
    "figure.figsize": (9.5, 4.2),          # wide — time needs horizontal room
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": "#cbcbcb", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": "#333333", "xtick.color": "#333333", "ytick.color": "#333333",
    "legend.frameon": False,
})

### 3. A first plot — and why it isn't done yet

A pandas Series plots itself with `.plot()`. Here's one Harmattan week (5–11
January 2026, selected with `.loc` exactly as in Notebook 3):

In [ ]:
week = pm.loc["2026-01-05":"2026-01-11"]
week.plot();

It *shows* the data, but it doesn't *say* anything: no units, no context, no
message. Three additions fix that:

1. a **y-axis label with units** — always `PM2.5 (µg/m³)`,
2. a **reference line** the reader can judge against (the WHO 24-hour guideline
   level, 15 µg/m³), labelled directly on the plot — no legend needed,
3. a **title that states the finding**, not the contents.

In [ ]:
ax = week.plot(color=GRAY)
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_xlabel("")
ax.set_ylim(0, None)
ax.axhline(15, color=BLUE, linestyle="--", linewidth=1)
ax.text(week.index[2], 20, "WHO 24-h guideline (15)", color=BLUE, fontsize=10)
ax.tick_params(axis="x", rotation=0)  # dates read best horizontal — never rotate labels
ax.set_title("A Harmattan week at Oshodi: PM2.5 never came near the WHO guideline");

### 4. The five-second test

The design doctrine behind that title (adapted from the AQ agent's chart-design
guide):

> **A chart exists to communicate one idea.** If someone looks at it for five
> seconds and cannot tell you what it's saying, the chart has failed.
>
> **Titles are not descriptions; they are assertions.**
> *Weak:* "PM2.5 levels by season" → *Strong:* "PM2.5 is 40% higher in the dry season"
> *Weak:* "Diurnal pattern of PM2.5" → *Strong:* "PM2.5 peaks at 8 AM, the morning rush"
>
> **Gray is the default; colour is the exception** — used only to draw the eye to
> the finding. If you highlight everything, you highlight nothing.

Run this cell to define the exercise checker:

In [ ]:
def check(name, test, hint=""):
    """Run test() and print a friendly ✅ or 💡 — never an error message."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    if ok:
        print(f"✅ {name} — looks right!")
    else:
        print(f"💡 {name} — not quite yet. Hint: {hint}")

def title_of(ax):
    """The axes title, wherever the house style put it (left or centre)."""
    return ax.get_title(loc="left") or ax.get_title()

✏️ **Your turn 4.1** — Make the same chart for a **rainy-season week**,
1–7 September 2025: select the week into `rainy_week`, plot it into `ax_rainy`
(gray line), add the y-axis unit label and the WHO reference line, and give it a
**finding title** (what does this week *say*? — compare the numbers with January's).

In [ ]:
# select the week with .loc, plot it in gray into ax_rainy, then add the
# ylabel, the WHO line, and a finding title (copy the pattern from section 3)
rainy_week = ...
ax_rainy = ...

In [ ]:
check("rainy_week covers the right dates",
      lambda: 100 < len(rainy_week) <= 168 and rainy_week.index[0].month == 9,
      'select with pm.loc["2025-09-01":"2025-09-07"]')
check("units on the y-axis", lambda: "µg/m³" in ax_rainy.get_ylabel(),
      'ax_rainy.set_ylabel("PM2.5 (µg/m³)")')
check("title states a finding", lambda: len(title_of(ax_rainy)) > 15,
      "write a sentence that asserts something, not just a description")

### 5. A whole year in one chart

Hourly data is too noisy to show a year. The Data-Explorer approach: **daily means
in light gray** for texture, and a **30-day rolling mean in the accent colour** to
carry the story. `.rolling(30, center=True)` averages each day with the 15 days
either side (`min_periods=15` tolerates our missing days):

In [ ]:
daily = pm.resample("D").mean()
smooth = daily.rolling(30, center=True, min_periods=15).mean()

ax = daily.plot(color=GRAY, linewidth=0.8, alpha=0.6)
smooth.plot(ax=ax, color=BLUE, linewidth=2.5)
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_xlabel("")
ax.set_ylim(0, None)
ax.text(daily.index[30], 8, "daily means", color=GRAY, fontsize=10)
ax.text(daily.index[140], 66, "30-day rolling mean", color=BLUE, fontsize=10)
ax.set_title("Harmattan lifts Oshodi PM2.5 roughly 60% above the April low");

Note the direct labels on the plot instead of a legend — the reader's eye never has
to leave the data.

### 6. Annotate the key event

The single worst day (you found it in Notebook 3: 13 January 2026) deserves to be
pointed at. `ax.annotate` draws a note with a thin arrow:

In [ ]:
ax = daily.plot(color=GRAY, linewidth=0.8, alpha=0.6)
smooth.plot(ax=ax, color=BLUE, linewidth=2.5)
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_xlabel("")
ax.set_ylim(0, None)
worst_day = daily.idxmax()
ax.annotate(f"worst day: {daily.max():.0f} µg/m³",
            xy=(worst_day, daily.max()), xytext=(worst_day, daily.max() + 12),
            fontsize=10, color="#555555", ha="center",
            arrowprops=dict(arrowstyle="-", color="#999999", linewidth=0.8))
ax.set_title("The year's worst air arrived with the January Harmattan peak");

✏️ **Your turn 4.2** — Your first full transfer task. The file
`abuja_pm25_2024.csv` holds calendar-2024 data from the **reference-grade monitor
at the US Embassy, Abuja** (the record ends 20 December, when the monitor went
offline). Build the year chart for Abuja:

1. load + datetime recipe → `abuja` (Abuja is also on West Africa Time),
2. daily means → `abuja_daily`, 30-day rolling → `abuja_smooth`,
3. plot gray daily + blue rolling into `ax_abuja`, unit label, and a finding title.

Fair warning: Abuja's Harmattan makes Lagos look mild. Let the title say so.

In [ ]:
abuja = ...
...
abuja_daily = ...
abuja_smooth = ...
ax_abuja = ...
...

In [ ]:
check("abuja_daily is a daily series", lambda: 340 < len(abuja_daily) < 370,
      'resample("D").mean() on the Abuja PM2.5 column gives ~355 days')
check("worst Abuja day found", lambda: 200 < abuja_daily.max() < 250,
      "the worst daily mean is around 226 µg/m³ (early February)")
check("rolling mean computed", lambda: abuja_smooth.notna().sum() > 250,
      "rolling(30, center=True, min_periods=15).mean() on abuja_daily")
check("units + a finding title", lambda: "µg/m³" in ax_abuja.get_ylabel() and len(title_of(ax_abuja)) > 15,
      "set_ylabel with units, and a title that asserts the finding")

### 7. Saving a chart

`savefig` writes the current figure to a file — PNG at 150 dpi is right for slides
and reports. (In Colab, the file appears in the 📁 Files panel on the left, where
you can download it.)

In [ ]:
fig = ax.figure  # the year chart with the annotation, from section 6
fig.savefig("oshodi_year.png", dpi=150, bbox_inches="tight")
print("saved oshodi_year.png")

### 8. Recap

* `.plot()` on a Series gives a time-series line; the house-style cell makes every
  chart consistent.
* A chart needs **units**, a **reference line** the reader can judge against, and
  a **title that states the finding** — the five-second test.
* Year view = gray daily means + accent rolling mean, labels **directly on the
  data**, key event annotated.
* `savefig` exports report-ready PNGs.

**Next: Notebook 5 — Diurnal and Monthly Patterns**, the two signature charts of
air-quality analysis: what time of day, and what time of year.